# Pipeline — SVM (Máquina de Soporte Vectorial)
### Módulo 3 · Tema 4.2

---
## ¿Qué hace SVM?
Encuentra el **hiperplano** (línea/plano de separación) que **maximiza el margen**
entre las dos clases. Los puntos más cercanos al hiperplano se llaman **vectores de soporte**.

```
Clase A:  o  o  o
                   ← margen máximo ←
Clase B:  x  x  x
```

## Parámetros clave
```
kernel → qué forma tiene la frontera de decisión:
  'linear' → línea recta (más simple)
  'rbf'    → curva (Radial Basis Function, el más común)
  'poly'   → polinomial

C → penalización por errores:
  C alto  → acepta menos errores, frontera más compleja → riesgo de overfitting
  C bajo  → acepta más errores, frontera más simple → más generalizable

gamma → solo para 'rbf' y 'poly':
  'scale' → valor recomendado para empezar
  Alto    → frontera muy compleja (overfitting)
  Bajo    → frontera muy suave
```

## ⚠️ IMPORTANTE: SVM SIEMPRE necesita escalar
SVM es muy sensible a la escala de los datos. Sin escalar, los resultados son muy malos.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 1 — Imports
# ═══════════════════════════════════════════════════════════════
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm             import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import accuracy_score, confusion_matrix, classification_report

RUTA = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Machote'))
if RUTA not in sys.path:
    sys.path.append(RUTA)

print("Imports listos ✅")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 2 — Cargar y preparar datos
# ← CAMBIA ESTE BLOQUE por tu dataset
# ═══════════════════════════════════════════════════════════════
from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
Y = dataset.target
CLASES = list(dataset.target_names)

# Seleccionar features útiles
df_temp = X.copy()
df_temp['__Y__'] = Y
corrs = df_temp.corr()['__Y__'].drop('__Y__')
features_utiles = corrs[corrs.abs() >= 0.5].index.tolist()
X_red = X[features_utiles]

# Dividir
X_train, X_test, y_train, y_test = train_test_split(
    X_red, Y, test_size=0.25, stratify=Y, random_state=42
)

# Escalar — OBLIGATORIO para SVM
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Datos listos — Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Features: {len(features_utiles)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3 — Comparar kernels
# ═══════════════════════════════════════════════════════════════
kernels = ['linear', 'rbf', 'poly']
resultados_kernel = {}

for k in kernels:
    m = SVC(kernel=k, C=1.0, random_state=42)
    m.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, m.predict(X_test_sc))
    resultados_kernel[k] = acc
    print(f"  kernel='{k}': {acc:.4f} ({acc*100:.1f}%)")

mejor_kernel = max(resultados_kernel, key=resultados_kernel.get)
print(f"\nMejor kernel: '{mejor_kernel}'")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 4 — Probar valores de C
# ═══════════════════════════════════════════════════════════════
valores_C = [0.01, 0.1, 1, 10, 100]
acc_C_vals = []

for c in valores_C:
    m = SVC(kernel=mejor_kernel, C=c, random_state=42)
    m.fit(X_train_sc, y_train)
    acc_C_vals.append(accuracy_score(y_test, m.predict(X_test_sc)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(valores_C, acc_C_vals, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('C (escala log)')
ax.set_ylabel('Accuracy en test')
ax.set_title(f'Accuracy vs C (kernel={mejor_kernel})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

mejor_C = valores_C[acc_C_vals.index(max(acc_C_vals))]
print(f"Mejor C: {mejor_C}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 5 — Modelo final y evaluación
# ═══════════════════════════════════════════════════════════════
modelo = SVC(
    kernel=mejor_kernel,
    C=mejor_C,
    random_state=42
)
modelo.fit(X_train_sc, y_train)

Y_pred = modelo.predict(X_test_sc)
acc    = accuracy_score(y_test, Y_pred)

print(f"SVM (kernel={mejor_kernel}, C={mejor_C}) — Accuracy: {acc:.4f} ({acc*100:.1f}%)\n")
print(classification_report(y_test, Y_pred, target_names=CLASES))

# Matriz de confusión
cm = confusion_matrix(y_test, Y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[f'Pred {c}' for c in CLASES],
    yticklabels=[f'Real {c}' for c in CLASES],
    ax=ax
)
ax.set_title(f'Matriz de Confusión SVM — {acc*100:.1f}%')
plt.tight_layout()
plt.show()